In [1]:
using CMPSExcitations

In [92]:
# canonical basis
function projection_matrix1(D, M)
    dim_in = D^2 + D      # input: E (DxD) + F (D)
    dim_out = 2 * D^2     # output: W1, W2 (both DxD)
    P = zeros(ComplexF64, dim_out, dim_in)

    for j in 1:dim_in
        e = zeros(Float64, dim_in)
        e[j] = 1.0

        E = reshape(view(e, 1:D^2), D, D)
        F = view(e, D^2+1:dim_in)

        W1 = E 
        W2 = E + M * Diagonal(F) / M

        P[:, j] = vcat(vec(W1), vec(W2))
    end

    return P
end

function projection_matrix2(D, R)
    dim_in = D^2 + D      # input: E (DxD) + F (D)
    dim_out = 2 * D^2     # output: W1, W2 (both DxD)
    P = zeros(ComplexF64, dim_out, dim_in)

    Dr, M = eigen(R)

    for j in 1:dim_in
        e = zeros(Float64, dim_in)
        e[j] = 1.0

        dD1 = view(e, 1:D)
        dD2 = view(e, D+1:2*D)
        X = zeros(D, D)

        k = 2D + 1
        for i in 1:D, j in 1:D
            if i != j
                X[i, j] = e[k]
                k += 1
            end
        end

        W1 = M * ((X .* Dr - Dr' .* X) + Diagonal(dD1)) / M
        W2 = M * ((X .* Dr - Dr' .* X) + Diagonal(dD2)) / M

        P[:, j] = vcat(vec(W1), vec(W2))
    end

    return P
end

# canonical basis
function excitation_matrix(Heff, D)
    dim = 2 * D^2
    M = zeros(ComplexF64, dim, dim)

    for j in 1:dim
        e = zeros(ComplexF64, dim)
        e[j] = 1.0
        W1 = reshape(view(e, 1:D^2), D, D)
        W2 = reshape(view(e, D^2+1:dim), D, D)
        W1p, W2p = Heff((Constant(W1), Constant(W2)))
        M[:, j] = vcat(vec(W1p[]), vec(W2p[]))
    end

    return M
end

excitation_matrix (generic function with 1 method)

In [93]:
Hsingle_ll(c, μ) = ∫(∂ψ̂' * ∂ψ̂ - μ * ψ̂' * ψ̂ + c * (ψ̂')^2 * ψ̂^2, (-Inf, +Inf));
Hsingle(c, μ) = ∫(2 * ∂ψ̂' * ∂ψ̂ - 2 * μ * ψ̂' * ψ̂ + 4 * c * (ψ̂')^2 * ψ̂^2, (-Inf, +Inf));
Hcoupled(c, μ) = ∫(
    (∂ψ̂₁' * ∂ψ̂₁ - μ * ψ̂₁' * ψ̂₁ + c * (ψ̂₁')^2 * ψ̂₁^2 +
     ∂ψ̂₂' * ∂ψ̂₂ - μ * ψ̂₂' * ψ̂₂ + c * (ψ̂₂')^2 * ψ̂₂^2 +
     2 * c * (ψ̂₁') * (ψ̂₂') * ψ̂₂ * ψ̂₁), (-Inf, +Inf));

In [96]:
c, μ = 10., 5.
tol = 1e-10

Ds = [4, 8]
D = maximum(Ds)

HLL = Hsingle(c, μ)
@time stateLL = find_groundstate(Ds, HLL, YangGaudinCMPS, optalg=LBFGS(80; verbosity=1, maxiter=7000, gradtol=tol), gradtol=tol)
println("Energy density: ", expval(HLL.h, stateLL)[], "\n Particle density: ", expval(ψ̂' * ψ̂, stateLL)[], "\n Order parameter: ", expval(ψ̂, stateLL)[])

# -----
stateCLL = InfiniteCMPS(stateLL.Q, (stateLL.Rs[1], stateLL.Rs[1]));
HCLL = Hcoupled(c, μ)
println("Energy density: ", expval(HCLL.h, stateCLL)[], "\nParticle density: ", expval(ψ̂₁' * ψ̂₁ + ψ̂₂' * ψ̂₂, stateCLL)[], "\nDensity imbalance: ", expval(ψ̂₁' * ψ̂₁ - ψ̂₂' * ψ̂₂, stateCLL)[])

Optimizing D=4


┌ Info: YangGaudinCMPS ground state: initialization with e = 882.308678356007
└ @ CMPSKit /home/ashankar/Documents/PhDstuff/code/TensorNetworks/CMPSKit.jl/src/yanggaudin/groundstate.jl:106
┌ Info: LBFGS: converged after 222 iterations: f = -2.734747817523, ‖∇f‖ = 2.1421e-11
└ @ OptimKit /home/ashankar/.julia/packages/OptimKit/xpmbV/src/lbfgs.jl:138


D = 4 | YangGaudinCMPS{Constant{Matrix{Float64}}, 1}
  0.151110 seconds (1.03 M allocations: 47.194 MiB, 1.84% gc time)
---------------
Optimizing D=8


┌ Info: YangGaudinCMPS ground state: converged after 223 iterations: e = -2.734747817523, ‖∇e‖ = 2.1421e-11
└ @ CMPSKit /home/ashankar/Documents/PhDstuff/code/TensorNetworks/CMPSKit.jl/src/yanggaudin/groundstate.jl:118
┌ Info: YangGaudinCMPS ground state: initialization with e = -2.734747817574
└ @ CMPSKit /home/ashankar/Documents/PhDstuff/code/TensorNetworks/CMPSKit.jl/src/yanggaudin/groundstate.jl:106
┌ Info: LBFGS: converged after 367 iterations: f = -2.761265509087, ‖∇f‖ = 5.0144e-11
└ @ OptimKit /home/ashankar/.julia/packages/OptimKit/xpmbV/src/lbfgs.jl:138


D = 8 | YangGaudinCMPS{Constant{Matrix{Float64}}, 1}
  3.035553 seconds (3.59 M allocations: 330.005 MiB, 1.34% gc time)
---------------
  3.186787 seconds (4.62 M allocations: 377.206 MiB, 1.36% gc time)
Energy density: -2.7612655090871083
 Particle density: 0.43647721538000434
 Order parameter: -0.36173976585014334
Energy density: -2.7612655090869573
Particle density: 0.8729544307600051
Density imbalance: 0.0


┌ Info: YangGaudinCMPS ground state: converged after 368 iterations: e = -2.761265509087, ‖∇e‖ = 5.0144e-11
└ @ CMPSKit /home/ashankar/Documents/PhDstuff/code/TensorNetworks/CMPSKit.jl/src/yanggaudin/groundstate.jl:118


In [97]:
# common setup
p = 0 # momentum
space = InfiniteCMPSExcitationSpace(p, stateCLL, stateCLL)
R = stateCLL.Rs[1][]
M = eigen(R).vectors
D = size(M, 1) # R = MDᵣ/M
H = excitation_matrix(excitation_operator(HCLL, space), D);

In [98]:
P = projection_matrix1(D, M)
vals, vecs = eigen(P' * H * P, P' * P)
println(real.(vals[1:5]))

[-0.06269297676128187, 1.1885905017093132, 2.331656134644417, 3.9138282385908743, 5.0891061057120375]


gauge seems to matter?

In [99]:
P = Matrix(qr(projection_matrix1(D, M)).Q)
vals, vecs = eigen(P' * H * P)
println(real.(vals[1:5]))

[-0.06269297676132794, 1.1885905017092793, 2.3316561346446023, 3.913828238590891, 5.089106105712137]


In [100]:
P = projection_matrix2(D, R)
vals, vecs = eigen(P' * H * P, P' * P)
println(real.(vals[1:5]))

[0.616356122243018, 2.839659755226983, 4.48834803223622, 6.168583936941106, 7.335363276342605]


In [101]:
P = Matrix(qr(projection_matrix2(D, R)).Q)
vals, vecs = eigen(P' * H * P)
println(real.(vals[1:5]))

[0.15855245794636114, 3.2566101895273807, 5.118511952836826, 6.2655793122529095, 7.136393154108954]


QR instead of geneigsolve seems to be consistently equivalent as expected. However, different parametrization does not seem to agree. Further, the same parametrization gives different results based on the ground state which presumably only varies by gauge.. ???